# 🏥 Notebook 02: 3D nnU-Net Baseline Training & Low-Data Benchmark
### SOTA Supervised Volumetric Convolutional Baseline (DynUNet with Deep Supervision)

This dedicated notebook runs the **3D nnU-Net Baseline**:
1. **Supervised 3D DynUNet Training (30 Epochs, AMP)**: MONAI DynUNet with residual units and multi-scale deep supervision heads at native and downsampled scales ($128^3, 64^3, 32^3, 16^3$). Loss weights dynamically follow normalized exponential decay $w_s = 2^{-s} / \sum_{j=0}^{S-1} 2^{-j}$ to support arbitrary head counts.
2. **Full-Data Held-Out Test Evaluation**: 3D Dice, IoU, 95th Percentile Hausdorff Distance (mm), and latency. Deep supervision heads are automatically handled during evaluation.
3. **Low-Data Volumetric Label Efficiency**: Evaluates performance degradation across annotation fractions ($1\%$ to $100\%$).
4. **Artifact Export**: Bundles checkpoints and metrics into `nnunet_outputs.zip`.

> **Estimated Runtime**: ~3.0 - 3.5 hours on NVIDIA Tesla T4 GPU (Well within Kaggle's 12-hour session limit).
> **Pro Tip**: You can run this notebook **in parallel** with Notebook 01 on a separate Kaggle GPU session!


## 1. Hardware & CUDA Environment Verification


In [ ]:
import datetime
import time

NOTEBOOK_START_TIME = time.time()
NOTEBOOK_START_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"⏱️ Session Start Time: {NOTEBOOK_START_STR}")

!nvidia-smi

import torch

print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(
        "WARNING: No GPU detected. Please navigate to Notebook Settings -> Accelerator -> GPU T4!"
    )


In [ ]:
# =========================================================================
# ⚙️ Experiment Configuration & Reproducibility Parameters
# =========================================================================

SEED = 42                 # Deterministic seed for weight init, augmentations, and data splits
NUM_WORKERS = 4          # Parallel CPU data loading workers (Kaggle T4 provides 4 vCPUs)
BATCH_SIZE = 2           # Volumetric batch size (nnU-Net consumes ~7.4 GB VRAM at B=2 with Deep Supervision)


## 2. Dependencies Installation


In [ ]:
!pip install -q --no-cache-dir monai nibabel tabulate matplotlib
import monai
import nibabel as nib
import tabulate

print(f"✓ MONAI Version:    v{monai.__version__}")
print(f"✓ NiBabel Version:  v{nib.__version__}")
print(f"✓ Tabulate Version: v{tabulate.__version__}")

## 3. Codebase Setup & Editable Installation


In [ ]:
import os
import shutil
import sys
from pathlib import Path

# Setup working directory in /kaggle/working
REPO_URL = "https://github.com/hanriman/tumor_segmentation_3d.git"
work_dir = Path("/kaggle/working/tumor_segmentation_3d")

# Clean up broken or incomplete clone from previous failed runs
if work_dir.exists() and not (work_dir / "src" / "brats_jepa_3d").exists():
    print("⚠️ Cleaning up incomplete repository clone...")
    shutil.rmtree(work_dir)

thesis_repo = Path("/kaggle/working/thesis_repo")
if thesis_repo.exists() and not (thesis_repo / "src" / "brats_jepa_3d").exists():
    shutil.rmtree(thesis_repo)

# Clone repository if not already present
if not (work_dir / "src" / "brats_jepa_3d").exists():
    if (thesis_repo / "src" / "brats_jepa_3d").exists():
        work_dir = thesis_repo
    elif Path("/kaggle/working/src/brats_jepa_3d").exists():
        work_dir = Path("/kaggle/working")
    else:
        print(f"Cloning codebase from: {REPO_URL} ...")
        !git clone {REPO_URL} {work_dir}

# Verify package was successfully cloned
src_dir = work_dir / "src"
if not (src_dir / "brats_jepa_3d").exists():
    raise RuntimeError(
        "❌ Clone failed! The package 'brats_jepa_3d' was not found on disk.\n"
        "👉 Please ensure 'Internet' is toggled ON in the Kaggle notebook settings (right sidebar)!"
    )

# Change working directory and update sys.path
os.chdir(str(work_dir))
%cd {work_dir}

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Install in editable mode
!pip install -q -e .

print(f"\n✓ Working directory set to: {Path.cwd()}")
!git log -1 --oneline 2>/dev/null || echo "(Git commit info unavailable)"


## 4. Dataset Discovery & Health Checks
Verifies whether processed `.npz` volumes exist. If absent, automatically invokes `prepare_data_3d.py` with multi-worker parallel resampling (`--num_workers 4`) and compact `float16` storage (~6.7 MB per volume, upcast to FP32 in RAM upon loading).


In [ ]:
import pandas as pd
import torch

from brats_jepa_3d.config import get_dataset_dir, get_metadata_path
from brats_jepa_3d.data import BraTS3DDataset

print("=== DATASET DISCOVERY ===")
data_dir = get_dataset_dir("brats_gli_3d")
meta_path = get_metadata_path("brats_gli_3d")
print(f"Dataset Directory: {data_dir} (Exists: {data_dir.exists()})")
print(f"Metadata CSV:      {meta_path} (Exists: {meta_path.exists()})")

# If preprocessed dataset is not found, check for raw BraTS data and run prepare_data_3d.py
if not meta_path.exists():
    print("\n⚠️ Preprocessed dataset not found. Running 3D preprocessing from raw BraTS data...")
    !python scripts/prepare_data_3d.py --limit 100 --dtype float16 --num_workers {NUM_WORKERS} --seed {SEED}
    meta_path = get_metadata_path("brats_gli_3d")

if meta_path.exists():
    df = pd.read_csv(meta_path)
    print(f"\n✓ Loaded Metadata: {len(df)} total records across splits:")
    print(df["split"].value_counts().to_string())

    ds = BraTS3DDataset(split="train")
    sample = ds[0]
    print("\n✓ Sample Tensor Verification:")
    print(f"  Image Shape: {sample['image'].shape} (dtype: {sample['image'].dtype})")
    print(f"  Mask Shape:  {sample['mask'].shape} (dtype: {sample['mask'].dtype})")
    print(f"  Tumor Voxels: {int((sample['mask'] > 0).sum()):,}")
else:
    print(
        "❌ ERROR: Please attach 'brats-3d-datasets' or raw BraTS dataset via '+ Add Input' in Kaggle!"
    )


## 5. Supervised 3D nnU-Net Training (30 Epochs, AMP)
Trains the self-configuring DynUNet architecture with intermediate supervision heads and dynamic exponential decay gradient weighting ($w_s = 2^{-s} / \sum_{j=0}^{S-1} 2^{-j}$, yielding $[w_0, w_1, w_2, w_3] \approx [0.533, 0.267, 0.133, 0.067]$ for 4 heads), prioritizing high-resolution segmentation while stabilizing early layer convergence.


In [ ]:
!python scripts/train_nnunet_3d.py \
    --seed {SEED} \
    --epochs 30 \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --learning_rate 3e-4 \
    --weight_decay 1e-4 \
    --amp


## 6. Held-Out Test Split Evaluation
Evaluates 3D nnU-Net on the held-out test split ($N=271$ volumes). Auxiliary deep supervision heads are automatically deactivated to evaluate deterministic full-resolution output.


In [ ]:
!python scripts/evaluate_3d.py \
    --model_type nnunet \
    --seed {SEED} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --amp


## 7. Low-Data Volumetric Label Efficiency for 3D nnU-Net
Evaluates supervised nnU-Net under extreme class imbalance and limited annotations ($1\%$ to $100\%$ labels).


In [ ]:
!python scripts/evaluate_low_data_3d.py \
    --model_type nnunet \
    --fractions 0.01 0.05 0.10 0.25 0.50 1.00 \
    --seed {SEED} \
    --epochs 15 \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --amp


## 8. Out-of-Distribution (OOD) Scanner Shift Robustness
Evaluates 3D nnU-Net under physical 3D Rician scanner noise ($\sigma=0.08$), quadratic RF $B_1$ field bias, and missing sequence triage.


In [ ]:
!python scripts/evaluate_ood_3d.py \
    --model_type nnunet \
    --seed {SEED} \
    --batch_size 1 \
    --num_workers {NUM_WORKERS} \
    --amp


## 9. Export & Package Artifacts
Packages nnU-Net checkpoints, training curves, and metrics into `nnunet_outputs.zip`.


In [ ]:
import datetime
import time
from pathlib import Path

from brats_jepa_3d.config import IN_KAGGLE, PROJECT_ROOT
from brats_jepa_3d.utils import export_artifacts

# 1. Package trained 3D nnU-Net checkpoints, metrics, and logs into a verified zip archive
base_working = Path("/kaggle/working") if IN_KAGGLE else PROJECT_ROOT
export_res = export_artifacts(
    export_name="nnunet_outputs",
    export_dir=base_working / "export_nnunet",
    zip_path=base_working / "nnunet_outputs.zip",
    model_prefix="nnunet",
    verbose=True,
)

# 2. Session Timing Report
NOTEBOOK_END_TIME = time.time()
NOTEBOOK_END_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
start_time = globals().get("NOTEBOOK_START_TIME", NOTEBOOK_END_TIME)
total_elapsed_sec = NOTEBOOK_END_TIME - start_time
hours, rem = divmod(total_elapsed_sec, 3600)
minutes, seconds = divmod(rem, 60)

print("\n" + "=" * 50)
print(f"⏱️ Session Start Time:   {globals().get('NOTEBOOK_START_STR', 'N/A')}")
print(f"⏱️ Session End Time:     {NOTEBOOK_END_STR}")
print(f"⏱️ Total Execution Time: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({total_elapsed_sec:.2f}s)")
print("=" * 50)
